# Historical Baltic extent and thickness

This notebook redraws the historical time series from the small processed tables included with the project. It does not need the gridded source archive.

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def find_root(start=Path.cwd()):
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "pyproject.toml").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Start Jupyter from this repository.")

ROOT = find_root()
DATA = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"
summary = pd.read_csv(DATA / "historical_baltic_winter_summary.csv")
chart_ts = pd.read_csv(DATA / "historical_baltic_chart_timeseries.csv", parse_dates=["date"])

def theil_sen_fit(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    keep = np.isfinite(x) & np.isfinite(y)
    x, y = x[keep], y[keep]
    if len(x) < 3:
        return np.nan, np.nan
    i, j = np.triu_indices(len(x), k=1)
    slope = float(np.median((y[j] - y[i]) / (x[j] - x[i])))
    return slope, float(np.median(y - slope * x))


In [ ]:
# Raw chart-date observations plus one winter-mean marker per complete season.
chart_ts=pd.read_csv(DATA/'historical_baltic_chart_timeseries.csv',parse_dates=['date'])
complete=summary[summary.complete_winter].copy()
era_style={'weekly':('#276A8A','Weekly'),'daily':('#C85C3D','Daily')}
metric_specs=[
    (None,'SIC-derived extent at 15%','thousand km$^2$',1000.0),
    ('mean_ice_thickness_cm','Mean thickness over mapped ice','cm',1.0),
]
fig,(ax_extent,ax_thickness)=plt.subplots(2,1,figsize=(15,8),sharex=True,constrained_layout=True)
trend_rows=[]
for era,(color,label) in era_style.items():
    raw=chart_ts[chart_ts.era==era]
    ax_extent.scatter(raw.date,raw.extent_15_km2/1000,s=5,alpha=.20,color=color,label=f'{label} charts (>=80% mapped area)',rasterized=True,zorder=1)
    thick=raw[np.isfinite(raw.thickness_mean_cm)]
    ax_thickness.scatter(thick.date,thick.thickness_mean_cm,s=5,alpha=.20,color=color,label=f'{label} charts (>=80% mapped area)',rasterized=True,zorder=1)
    seasonal=complete[complete.era==era]
    center_dates=pd.to_datetime((seasonal.winter_start_year+1).astype(str)+'-02-15')
    for ax,mean_col,scale,unit in (
        (ax_extent,'mean_sampled_extent_15_km2',1000.0,'thousand km$^2$'),
        (ax_thickness,'mean_ice_thickness_cm',1.0,'cm'),
    ):
        values=seasonal[mean_col].to_numpy(dtype=float)
        years=seasonal.winter_start_year.to_numpy(dtype=float)
        slope,intercept=theil_sen_fit(years,values)
        slope_per_decade=slope*10/scale
        ax.scatter(center_dates,values/scale,s=42,marker='D',color=color,edgecolor='white',linewidth=.6,
                   label=f'{label} winter mean (trend {slope_per_decade:+.1f} {unit}/decade)',zorder=4)
        x_year=np.array([years.min(),years.max()])
        x_date=pd.to_datetime((x_year.astype(int)+1).astype(str)+'-02-15')
        ax.plot(x_date,(intercept+slope*x_year)/scale,color=color,ls='--',lw=1.8,zorder=3)
        trend_rows.append({'metric':mean_col,'era':era,'first_winter':int(years.min()),
            'last_winter':int(years.max()),'n_winters':len(years),
            'theil_sen_slope_per_decade':slope_per_decade,
            'units_per_decade':unit.replace('$^2$','^2')+'/decade'})

# Show available records from the incomplete current winter without a seasonal mean or trend.
partial=chart_ts[chart_ts.era=='partial']
if not partial.empty:
    ax_extent.scatter(partial.date,partial.extent_15_km2/1000,s=6,alpha=.25,color='#777777',label='Partial 2025/26 charts (>=80% mapped area)',rasterized=True,zorder=1)
    pt=partial[np.isfinite(partial.thickness_mean_cm)]
    ax_thickness.scatter(pt.date,pt.thickness_mean_cm,s=6,alpha=.25,color='#777777',label='Partial 2025/26 charts (>=80% mapped area)',rasterized=True,zorder=1)

for ax in (ax_extent,ax_thickness):
    ax.axvline(pd.Timestamp('2006-12-01'),color='#555',ls=':',lw=.9)
    ax.grid(axis='y',alpha=.25)
    ax.legend(frameon=False,ncol=1,fontsize=8,loc='upper left',bbox_to_anchor=(1.005,1.0),borderaxespad=0)
ax_extent.set_ylabel('thousand km$^2$')
ax_extent.set_title('Coverage-screened chart-date extent and winter mean',loc='left',fontsize=12)
ax_thickness.set_ylabel('cm')
ax_thickness.set_title('Coverage-screened chart-date thickness and winter mean',loc='left',fontsize=12)
ax_thickness.set_xlabel('Date')
ax_thickness.xaxis.set_major_locator(mdates.YearLocator(5))
ax_thickness.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
fig.suptitle('Historical Baltic basin extent and ice thickness | chart observations and winter means',fontsize=14,fontweight='bold')
trend_summary=pd.DataFrame(trend_rows)
trend_path=DATA/'historical_baltic_trend_summary.csv'
trend_summary.to_csv(trend_path,index=False)
series_fig=FIGURES/'historical_baltic_extent_and_thickness.png'
fig.savefig(series_fig,dpi=180,bbox_inches='tight',facecolor='white'); plt.show()
print(trend_summary.round(2).to_string(index=False))
print('Chart-date data:',DATA/'historical_baltic_chart_timeseries.csv')
print('Trend table:',trend_path)
print('Figure:',series_fig)